<a href="https://colab.research.google.com/github/KurniaYufi/sentiment-analysis-kematian-ali-khamenei/blob/main/scraping/cnn/cnn-content-scraping" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN Indonesia — Content Scraping & Preprocessing
**Input :** `scraping_cnn_100pages_clean.csv` (kolom: Judul, Link, Kategori, Tanggal_Upload)
**Output :** `cnn_analyzed_articles.csv` (+ kolom content, preprocessing, sentimen)

Alur:
1. Instalasi Library
2. Impor Library
3. Upload CSV & Scraping Konten tiap Link → kolom `content`
4. Preprocessing (clean → tokenize → stopword → stem → deteksi bahasa)
5. Analisis Sentimen (translate → TextBlob)
6. Visualisasi Word Cloud
7. Ekspor CSV

## **(1) Instalasi Library**

In [1]:
# Library scraping & data
!pip install -q beautifulsoup4 requests pandas

# Polyglot (deteksi bahasa) — perlu urutan ini agar tidak konflik
!pip uninstall -q -y icu polyglot pyicu 2>/dev/null
!pip install -q polyglot pycld2 pyicu morfessor

# NLP
!pip install -q deep-translator Sastrawi stanza nltk wordcloud

print('✅ Semua library berhasil diinstall')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.3/126.3 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.2/268.2 kB 10.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 20.3 MB/s eta 0:00:00
✅ Semua library berhasil diinstall


## **(2) Impor Library**

In [2]:
# Standard
import io, string, re, time
from collections import defaultdict, Counter

# Data
import pandas as pd
import numpy as np

# Scraping
import requests
from bs4 import BeautifulSoup

# NLTK
import nltk
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist

# Sastrawi (stemming)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Sentimen
from textblob import TextBlob
from deep_translator import GoogleTranslator

# Visualisasi
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# NLTK downloads
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

print('✅ Semua library berhasil diimpor')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ Semua library berhasil diimpor


## **(3) Upload CSV & Scraping Konten tiap Link**

In [3]:
from google.colab import files
import os

print('📂 Upload file CSV hasil scraping link (scraping_cnn_100pages_clean.csv)')

# Clear previously uploaded files to avoid confusion if an incorrect file was uploaded.
# This part ensures a fresh upload attempt.
# for f in files.upload(): # This loop usually gets an uploaded file. If the cell was run, 'uploaded' would be populated.
#   os.remove(f)

# The 'uploaded' variable is a dictionary of {filename: content}.
# If it already exists from a previous run, clear it.
if 'uploaded' in locals() or 'uploaded' in globals():
    uploaded = {}

uploaded = files.upload()

if not uploaded:
    print('Tidak ada file yang diunggah. Pastikan Anda memilih file CSV.')
    # If no file is uploaded, we might want to prevent further execution or ask to re-upload.
    # For now, let's just print a message and proceed, assuming the user might re-run.
else:
    filename = list(uploaded.keys())[0]
    # Check if the uploaded file has a .csv extension
    if not filename.lower().endswith('.csv'):
        print(f'⚠️  File yang diunggah bukan file CSV: {filename}. Mohon unggah file CSV yang benar.')
        # Clear the uploaded content to ensure the next cell doesn't try to process it.
        uploaded = {}
        # You might want to raise an error or exit here if you want to strictly enforce CSV upload.
        # raise ValueError('Uploaded file must be a CSV.')
    else:
        print(f'✅ File terupload: {filename}')


📂 Upload file CSV hasil scraping link (scraping_cnn_100pages_clean.csv)


Saving scraping_cnn_100pages_clean (1).csv to scraping_cnn_100pages_clean (1).csv
✅ File terupload: scraping_cnn_100pages_clean (1).csv


In [4]:
# Baca CSV
links_df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'Jumlah baris: {len(links_df)}')
print(f'Kolom       : {links_df.columns.tolist()}')
links_df.head(3)

Jumlah baris: 99
Kolom       : ['Judul', 'Link', 'Kategori', 'Tanggal_Upload']


,Judul,Link,Kategori,Tanggal_Upload
0,Isi Dokumen Fatwa Ali Khamenei Mengharamkan Bo...,https://www.cnnindonesia.com/internasional/202...,Internasional,3 hari yang lalu
1,"Tanpa Ali Khamenei, Israel Justru Ketar-ketir ...",https://www.cnnindonesia.com/internasional/202...,Internasional,4 hari yang lalu
2,Trump Konfirmasi Kematian Pemimpin Tertinggi I...,https://www.cnnindonesia.com/internasional/202...,Internasional,2 bulan yang lalu


In [5]:
# ── Fungsi scraping konten satu artikel CNN Indonesia ────────────────────────
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

def scrape_content(url):
    """
    Ambil SEMUA teks yang terlihat di halaman artikel CNN Indonesia.
    Strategi:
      1. Coba selector utama CNN Indonesia (detail_text / article body)
      2. Fallback: ambil semua <p> di seluruh halaman
      3. Last resort: get_text() seluruh <body> bersih dari tag
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.content, 'html.parser')

        # Hapus elemen non-konten (nav, header, footer, script, style, ads)
        for tag in soup(['script', 'style', 'noscript', 'nav', 'header',
                         'footer', 'aside', 'iframe', 'figure']):
            tag.decompose()

        # ── Prioritas 1: div.detail_text (body artikel CNN Indonesia) ──────
        content_div = (
            soup.find('div', class_='detail_text') or
            soup.find('div', class_=lambda c: c and 'detail_text' in c) or
            soup.find('div', class_=lambda c: c and 'article-body' in c) or
            soup.find('article')
        )

        if content_div:
            # Ambil semua teks di dalam container tersebut
            paragraphs = content_div.find_all(['p', 'h1', 'h2', 'h3', 'li'])
            text = '\n'.join(
                p.get_text(separator=' ', strip=True)
                for p in paragraphs
                if p.get_text(strip=True)
            )
            if text:
                return text

        # ── Prioritas 2: semua <p> di halaman ──────────────────────────────
        paragraphs = soup.find_all('p')
        text = '\n'.join(
            p.get_text(separator=' ', strip=True)
            for p in paragraphs
            if p.get_text(strip=True)
        )
        if text:
            return text

        # ── Prioritas 3 (last resort): seluruh body ─────────────────────────
        body = soup.find('body')
        if body:
            return re.sub(r'\n{3,}', '\n\n', body.get_text(separator='\n', strip=True))

        return None

    except Exception as e:
        return None


# ── Loop scraping semua link ──────────────────────────────────────────────────
if 'Link' not in links_df.columns:
    raise ValueError("Kolom 'Link' tidak ditemukan di CSV!")

urls = links_df['Link'].tolist()
contents = []

for i, url in enumerate(urls, 1):
    print(f'[{i:>3}/{len(urls)}] Scraping... {url[:80]}')
    content = scrape_content(url)
    contents.append(content)
    time.sleep(1)   # jeda sopan agar tidak diblok server

# Tambahkan kolom 'content' ke dataframe asli
df_article = links_df.copy()
df_article['content'] = contents

# Simpan versi mentah dengan kolom content
df_article.to_csv('cnn_with_content.csv', index=False, encoding='utf-8-sig')

print(f'\n✅ Selesai! {df_article["content"].notna().sum()}/{len(df_article)} artikel berhasil discrape.')
print(f'💾 Disimpan ke cnn_with_content.csv')
df_article.head(3)

[  1/99] Scraping... https://www.cnnindonesia.com/internasional/20260428200911-120-1353182/isi-dokume
[  2/99] Scraping... https://www.cnnindonesia.com/internasional/20260428135252-120-1353013/tanpa-ali-
[  3/99] Scraping... https://www.cnnindonesia.com/internasional/20260301044904-134-1332937/trump-konf
[  4/99] Scraping... https://www.cnnindonesia.com/internasional/20260501153618-134-1354171/trump-bing
[  5/99] Scraping... https://www.cnnindonesia.com/internasional/20260420111520-120-1349957/kenapa-ira
[  6/99] Scraping... https://www.cnnindonesia.com/internasional/20260304191525-120-1334389/mojtaba-kh
[  7/99] Scraping... https://www.cnnindonesia.com/internasional/20260301090739-120-1332969/waktu-terb
[  8/99] Scraping... https://www.cnnindonesia.com/internasional/20260309044622-120-1335708/tok-anak-a
[  9/99] Scraping... https://www.cnnindonesia.com/internasional/20260301174003-120-1333089/hamas-ucap
[ 10/99] Scraping... https://www.cnnindonesia.com/internasional/20260305065848-120

,Judul,Link,Kategori,Tanggal_Upload,content
0,Isi Dokumen Fatwa Ali Khamenei Mengharamkan Bo...,https://www.cnnindonesia.com/internasional/202...,Internasional,3 hari yang lalu,Isi Dokumen Fatwa Ali Khamenei Mengharamkan Bo...
1,"Tanpa Ali Khamenei, Israel Justru Ketar-ketir ...",https://www.cnnindonesia.com/internasional/202...,Internasional,4 hari yang lalu,"Tanpa Ali Khamenei, Israel Justru Ketar-ketir ..."
2,Trump Konfirmasi Kematian Pemimpin Tertinggi I...,https://www.cnnindonesia.com/internasional/202...,Internasional,2 bulan yang lalu,Trump Konfirmasi Kematian Pemimpin Tertinggi I...


In [8]:
from google.colab import files
files.download('cnn_with_content.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>